In [ ]:
import json
import math
import time
import numpy as np
import pandas as pd
import os
from random import randint

import torch
import torch.optim as optim
import torch.utils.data as data
import matplotlib.pyplot as plt

import joblib

from utils.reproducibility import seed_everything, tf_func, clip_grad_func
from utils.loss_fn import masked_mae_multi as loss_fn
from utils.create_dataset_v1 import make_input_data, StationDataset
from utils.models import (AttrSeq2SeqLSTM as seq2seq_model, 
                          AttrSeq2SeqLSTM_v1 as seq2seq_model_v1,
                          AttrSeq2SeqCNNLSTM as seq2seq_cnn_model,
                          AttrLSTM as norm_model, 
                          AttrLSTMv1 as norm_model_v1, 
                          train_one_epoch, 
                          evaluate,
                          train_one_epoch_scaler,
                          evaluate_scaler
                          )
from utils.utils import read_forecast_data

from sklearn.model_selection import train_test_split

import ipywidgets as widgets
from IPython.display import display
from lstm1__const__ import DAYS



In [ ]:
try:
    print(TYPE)
except Exception:
    TYPE = next(iter(DAYS))
    ATTR = next(iter(DAYS[TYPE]))
    DAY = DAYS[TYPE][ATTR]

# DAYS = {
#   "LSTM-ED-FC0": {
#     "Temp": "LSTM_Temp_huber0_seq2seq_v1_fc",
#     "RH": "LSTM_RH_huber0_seq2seq_v1_fc",
#     "WSpd": "LSTM_WSpd_huber0_seq2seq_v1_fc",
#     "WDir": "LSTM_WDir_huber0_seq2seq_v1_fc"
#   },
#   "LSTM-ED": {
#     "Temp": "LSTM_Temp_mae_seq2seq_v1",
#     "RH": "LSTM_RH_mae_seq2seq_v1",
#     "WSpd": "LSTM_WSpd_mae_seq2seq_v1",
#     "WDir": "LSTM_WDir_mae_seq2seq_v1"
#   },
#   "LSTM-ED-FC": {
#     "Temp": "LSTM_Temp_mae_seq2seq_v1_fc",
#     "RH": "LSTM_RH_mae_seq2seq_v1_fc",
#     "WSpd": "LSTM_WSpd_mae_seq2seq_v1_fc",
#     "WDir": "LSTM_WDir_mae_seq2seq_v1_fc"
#   },
# }



In [ ]:

def get_prev_loss(model_folder):
    FILE_DIR = f'models/{model_folder}'
    prev_losses = None
    if not os.path.exists(FILE_DIR): return prev_losses
    MODEL_LOADED = None
    max_epoch = -1
    filename = None
    for i in os.listdir(FILE_DIR):
        epoch = int(i.split('epoch_')[1].split('.pth')[0])
        if max_epoch < epoch:
            max_epoch = epoch
            filename = f"{FILE_DIR}/{i}"
    if filename is not None:
        try:
            MODEL_LOADED = torch.load(filename, map_location='cpu', weights_only=False)
        except TypeError:
            MODEL_LOADED = torch.load(filename, map_location='cpu')
        extra = MODEL_LOADED['extra']
        prev_losses = MODEL_LOADED["prev_losses"]
    return prev_losses



In [ ]:
from matplotlib.lines import Line2D

log_scaling = False
ATTR_ORDER = ['Temp', 'RH', 'WSpd', 'WDir']

# one consistent color per model type
type_names = list(DAYS.keys())
cmap = plt.get_cmap('tab10')
type_colors = {t: cmap(i % 10) for i, t in enumerate(type_names)}

# One plot per attribute; overlay every model type (color = type, dashed = train, solid = test)
for attr in ATTR_ORDER:
    present = []
    for t in type_names:
        if attr in DAYS[t]:
            pl = get_prev_loss(DAYS[t][attr])
            if pl and 'mean' in pl.get('train', {}):
                present.append((t, pl))
    if not present:
        continue

    plt.figure(figsize=(15, 6))
    legend_labels = []
    for t, pl in present:
        train_loss = np.array(pl['train']['mean'])
        test_loss = np.array(pl['test']['mean'])
        if log_scaling:
            train_loss = np.log10(train_loss)
            test_loss = np.log10(test_loss)
        train_loss[0] = np.nan
        color = type_colors[t]
        plt.plot(train_loss, linestyle='--', color=color, alpha=0.6)
        plt.plot(test_loss, linestyle='-', color=color)
        best_ep = int(np.argmin(test_loss))
        legend_labels.append(f'{t} (best test {test_loss[best_ep]:#.4g} @ ep{best_ep})')

    # legend 1: color -> model type (+ its best test)
    color_handles = [Line2D([0], [0], color=type_colors[t], lw=2) for t, _ in present]
    leg1 = plt.legend(color_handles, legend_labels, title='Model', loc='upper right', fontsize=9)
    plt.gca().add_artist(leg1)
    # legend 2: line style -> train/test
    style_handles = [Line2D([0], [0], color='k', lw=2, linestyle='--'),
                     Line2D([0], [0], color='k', lw=2, linestyle='-')]
    plt.legend(style_handles, ['train', 'test'], title='Split', loc='lower left', fontsize=9)

    plt.title(f'{attr}  -  train (dashed) / test (solid)')
    plt.xlabel('Epoch')
    plt.ylabel('Loss' + (' (log10)' if log_scaling else ''))
    plt.grid(True)
    plt.show()
